In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, roc_auc_score, precision_score, recall_score, f1_score, accuracy_score
import tensorflow as tf
from keras.models import Sequential, load_model
from keras.optimizers import Adam
from datetime import datetime



Data Initialisation

In [ ]:
df = pd.read_csv('enriched_timeline.csv')
X_padded = np.load("X_padded.npy", allow_pickle=True)
y_padded = np.load("y_padded.npy", allow_pickle=True)
df.drop(columns=['last_act_date'], inplace=True)

In [ ]:
scaler = RobustScaler()
scaled_data = scaler.fit_transform(df[["y_next24h"]])

In [ ]:
from keras.preprocessing.sequence import pad_sequences
import numpy as np

X_padded = np.load("X_padded.npy", allow_pickle=True)
y_padded = np.load("y_padded.npy", allow_pickle=True)


patient_sequences = [X[~np.all(X == 0, axis=1)] for X in X_padded]
patient_labels = [y[~np.all(X == 0, axis=1)] for X, y in zip(X_padded, y_padded)]

def resample_sequences(patient_sequences, patient_labels):
    X_out, y_out = [], []

    for Xi, yi in zip(patient_sequences, patient_labels):
        X_out.append(Xi)
        seq_label = int(np.any(yi > 0))
        y_out.append(seq_label)

    X_padded = pad_sequences(X_out, padding="post", dtype="float32")

    y_out = np.array(y_out).reshape(-1, 1)

    return X_padded, y_out


X_padded, y_padded = resample_sequences(patient_sequences, patient_labels)

print("X_padded:", X_padded.shape, "| y_padded:", y_padded.shape)
print("Positives:", np.sum(y_padded), "| Negatives:", len(y_padded) - np.sum(y_padded))


In [ ]:
from sklearn.model_selection import train_test_split


patient_ids = np.array([pid for pid in df['Patient_ID'].unique()])

train_ids, test_ids = train_test_split(patient_ids, test_size=0.15, random_state=42)
train_ids, val_ids  = train_test_split(train_ids, test_size=0.176, random_state=42) 
# 0.176 ≈ 15% total, so 70/15/15

pid_order = list(df['Patient_ID'].unique())
pid_to_idx = {pid:i for i,pid in enumerate(pid_order)}

train_idx = [pid_to_idx[pid] for pid in train_ids]
val_idx   = [pid_to_idx[pid] for pid in val_ids]
test_idx  = [pid_to_idx[pid] for pid in test_ids]

X_train, y_train = X_padded[train_idx], y_padded[train_idx]
X_val, y_val     = X_padded[val_idx], y_padded[val_idx]
X_test, y_test   = X_padded[test_idx], y_padded[test_idx]


y_train = np.squeeze(y_train)
y_val   = np.squeeze(y_val)
y_test  = np.squeeze(y_test)

if y_train.ndim > 1:
    y_train = y_train[:, -1]
    y_val   = y_val[:, -1]
    y_test  = y_test[:, -1]

y_train = y_train.reshape(-1, 1)
y_val   = y_val.reshape(-1, 1)
y_test  = y_test.reshape(-1, 1)

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)



In [ ]:
import numpy as np

balanced_X = []
balanced_y = []

pos_sequences = [(X_seq, y_seq) for X_seq, y_seq in zip(X_train, y_train) if np.any(y_seq == 1)]
neg_sequences = [(X_seq, y_seq) for X_seq, y_seq in zip(X_train, y_train) if np.all(y_seq == 0)]

print(f"Original training set: {len(X_train)} sequences")
print(f" → Positive sequences: {len(pos_sequences)}")
print(f" → Negative sequences: {len(neg_sequences)}")

for X_seq, y_seq in pos_sequences:
    balanced_X.append(X_seq)
    balanced_y.append(y_seq)

pos_count = len(pos_sequences)
if len(neg_sequences) < pos_count:
    print("Fewer negative sequences than positives. Sampling negatives with replacement.")
    sampled_neg_idx = np.random.choice(len(neg_sequences), size=pos_count, replace=True)
else:
    sampled_neg_idx = np.random.choice(len(neg_sequences), size=pos_count, replace=False)

for idx in sampled_neg_idx:
    balanced_X.append(neg_sequences[idx][0])
    balanced_y.append(neg_sequences[idx][1])

X_train = np.array(balanced_X, dtype="float32")
y_train = np.array(balanced_y, dtype="float32")

pos_after = sum(np.any(seq == 1) for seq in y_train)
neg_after = sum(np.all(seq == 0) for seq in y_train)

print("\nBalanced training set:")
print(f" → Total sequences: {len(y_train)}")
print(f" → Positive sequences: {pos_after}")
print(f" → Negative sequences: {neg_after}")
print(f" → Class ratio: {pos_after/(pos_after+neg_after):.2f} positive / {neg_after/(pos_after+neg_after):.2f} negative")


In [ ]:
from sklearn.model_selection import train_test_split


patient_ids = np.array([pid for pid in df['Patient_ID'].unique()])

train_ids, test_ids = train_test_split(patient_ids, test_size=0.15, random_state=42)
train_ids, val_ids  = train_test_split(train_ids, test_size=0.176, random_state=42) 
# 0.176 ~ 15% total, so 70/15/15

pid_order = list(df['Patient_ID'].unique())
pid_to_idx = {pid:i for i,pid in enumerate(pid_order)}

train_idx = [pid_to_idx[pid] for pid in train_ids]
val_idx   = [pid_to_idx[pid] for pid in val_ids]
test_idx  = [pid_to_idx[pid] for pid in test_ids]

X_train, y_train = X_padded[train_idx], y_padded[train_idx]
X_val, y_val     = X_padded[val_idx], y_padded[val_idx]
X_test, y_test   = X_padded[test_idx], y_padded[test_idx]


y_train = np.squeeze(y_train)
y_val   = np.squeeze(y_val)
y_test  = np.squeeze(y_test)

if y_train.ndim > 1:
    y_train = y_train[:, -1]
    y_val   = y_val[:, -1]
    y_test  = y_test[:, -1]

y_train = y_train.reshape(-1, 1)
y_val   = y_val.reshape(-1, 1)
y_test  = y_test.reshape(-1, 1)

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)



XAI

In [ ]:
from keras.models import load_model
model = load_model("GRU 3-9 2.h5")

print(model.summary())


In [ ]:
import shap

feature_names = list(df.columns)

# one vector per patient
X_train_agg = X_train.mean(axis=1)
X_test_agg  = X_test.mean(axis=1)

print("Aggregated shapes:", X_train_agg.shape, X_test_agg.shape)

def predict_fn(x):
    x_expanded = np.repeat(x[:, None, :], X_train.shape[1], axis=1)
    return model.predict(x_expanded).flatten()

background = X_train_agg[:10]
test_sample = X_test_agg[:5]
explainer = shap.KernelExplainer(predict_fn, background)
shap_values = explainer.shap_values(test_sample, nsamples=50)
shap_feature_importance = np.abs(shap_values).mean(axis=0)
n_features = shap_feature_importance.shape[0]
if len(feature_names) != n_features:
    print(f"Adjusting feature_names: expected {n_features}, got {len(feature_names)}")
    feature_names = feature_names[:n_features]


sorted_idx = np.argsort(shap_feature_importance)[::-1]
print("\nTop features by SHAP importance:")
for i in range(n_features):
    feat = feature_names[sorted_idx[i]]
    score = shap_feature_importance[sorted_idx[i]]
    print(f"{i+1:2d}. {feat:25s} {score:.6f}")

plt.figure(figsize=(10, 6))
plt.bar(range(n_features), shap_feature_importance[sorted_idx])
plt.xticks(range(n_features), [feature_names[i] for i in sorted_idx],
           rotation=45, ha="right")
plt.ylabel("Mean |SHAP value|")
plt.title("Global Feature Importance (SHAP)")
plt.tight_layout()
plt.show()



In [ ]:
import tf_keras_vis
print(tf_keras_vis.__version__)


In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import PartialDependenceDisplay
from sklearn.metrics import r2_score
from scipy.stats import pearsonr

X_train_agg = X_train.mean(axis=1)
X_test_agg  = X_test.mean(axis=1)

print("Aggregated shapes:", X_train_agg.shape, X_test_agg.shape)

y_pred_train = model.predict(
    np.repeat(X_train_agg[:, None, :], X_train.shape[1], axis=1),
    verbose=0
).ravel()

y_pred_test = model.predict(
    np.repeat(X_test_agg[:, None, :], X_train.shape[1], axis=1),
    verbose=0
).ravel()

surrogate = RandomForestRegressor(n_estimators=50, random_state=42)
surrogate.fit(X_train_agg, y_pred_train)

surrogate_pred_test = surrogate.predict(X_test_agg)

r2 = r2_score(y_pred_test, surrogate_pred_test)
corr, _ = pearsonr(y_pred_test, surrogate_pred_test)

print(f"Surrogate fidelity on test set:")
print(f" - R² score: {r2:.3f}")
print(f" - Pearson correlation: {corr:.3f}")

top_features = [
    # "any_death",
    "any_obs_event",
    "num_prn_72h",
    "restraint_physical_aggression",
    "time_since_last_activity_hours",
    "activity_today",
    "any_safeguarding"
]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
PartialDependenceDisplay.from_estimator(
    surrogate,
    X_test_agg,
    features=top_features,
    feature_names=feature_names,
    kind="average",
    grid_resolution=30,
    ax=axes
)
fig.suptitle("Partial Dependence Plots (Surrogate model)", fontsize=16)
plt.tight_layout()
plt.show()


Widget with Explanations


In [ ]:
from keras.models import load_model
model = load_model("GRU 3-9 2.h5")

print(model.summary())

In [ ]:
import ipywidgets as widgets
from IPython.display import display

def predict_patient(i):
    x_sample = X_test[i:i+1]
    prob = model.predict(x_sample, verbose=0)[0][0]
    print(f"Patient {i} - Predicted risk of aggression in next 24h: {prob:.2%}")

slider = widgets.IntSlider(min=0, max=len(X_test)-1, step=1, value=0)
widgets.interact(predict_patient, i=slider)


In [ ]:
feature_cols = [
    'on_antipsychotic',
    'num_prn_72h',
    'activity_today',
    'time_since_last_activity_hours',
    'Age'
] + [c for c in df.columns if c.startswith('Gender_')] + [
    'aggression_today',
    'any_restraint',
    'any_selfharm',
    'any_safeguarding',
    'any_obs_event',
    'restraint_physical_aggression'
]

feat_names = list(feature_cols)
print("Feature columns used:", feat_names)

In [ ]:
import numpy as np, matplotlib.pyplot as plt, shap, ipywidgets as widgets
from IPython.display import display
from functools import lru_cache

def _to_np(x): 
    import tensorflow as tf
    if tf.is_tensor(x):
        try: return x.numpy()
        except Exception: return tf.convert_to_tensor(x).numpy()
    return np.asarray(x)

X_train = _to_np(X_train).astype(np.float32)
X_test  = _to_np(X_test).astype(np.float32)
assert X_train.ndim == 3 and X_test.ndim == 3, "Expected (N, T, F) arrays."
T, F = X_test.shape[1], X_test.shape[2]
feat_names = list(feature_cols)
assert len(feat_names) == F, f"feature_cols length {len(feature_cols)} != F={F}"

def _predict_from_flat(X_flat):
    X_flat = np.asarray(X_flat, dtype=np.float32)
    if X_flat.ndim == 1:
        X_flat = X_flat[None, :]
    B = X_flat.shape[0]
    x = X_flat.reshape(B, T, F)
    return np.asarray(model.predict(x, verbose=0)).reshape(-1)

def _flatten(x):
    xx = np.asarray(x, dtype=np.float32)
    return xx.reshape(xx.shape[0], -1)

@lru_cache(maxsize=1)
def _get_kernel_explainer(bg_size=24):
    n = min(bg_size, len(X_train))
    bg_idx = np.random.choice(len(X_train), size=n, replace=False)
    bg_flat = _flatten(X_train[bg_idx])          # (n, T*F)
    expl = shap.KernelExplainer(_predict_from_flat, bg_flat)
    return expl

def explain_sample(i, aggregate="signed", nsamples=100):
    x = X_test[i:i+1]                 # (1, T, F)
    prob = float(model.predict(x, verbose=0)[0][0])

    expl = _get_kernel_explainer()
    sv = expl.shap_values(_flatten(x), nsamples=nsamples)
    if isinstance(sv, list):
        sv = sv[0]
    sv = np.asarray(sv, dtype=np.float32).reshape(T, F)

    if aggregate == "magnitude":
        feat_contrib = np.abs(sv).sum(axis=0)
    else:  # signed +/- risk
        feat_contrib = sv.sum(axis=0)

    return prob, feat_contrib, sv


def predict_and_explain(i, top_k=8, aggregation="signed"):
    prob, feat_contrib, shap_2d = explain_sample(i, aggregate=aggregation)

    order = np.argsort(np.abs(feat_contrib))[::-1][:top_k]
    top_feats = [feat_names[j] for j in order]
    top_vals  = feat_contrib[order]

    plt.figure(figsize=(6, 4))
    plt.title(f"Predicted risk (next 24h): {prob:.2%}")
    plt.axis("off")
    plt.show()

    plt.figure(figsize=(7, max(3, 0.35*top_k)))
    y = np.arange(len(top_feats))
    plt.barh(y, top_vals, align="center")
    plt.yticks(y, top_feats)
    plt.xlabel("Contribution" if aggregation=="signed" else "Importance (|SHAP|)")
    plt.title(f"Top {top_k} features ({aggregation})")
    plt.gca().invert_yaxis()
    plt.grid(True, axis="x", linestyle="--", alpha=0.35)
    plt.tight_layout()
    plt.show()

    # Time × feature heatmap for top features
    # sub = shap_2d[:, order]  # (T, top_k)
    # plt.figure(figsize=(8, max(3, 0.35*top_k)))
    # im = plt.imshow(sub.T, aspect="auto", interpolation="nearest")
    # plt.colorbar(im, label="SHAP value (signed)")
    # plt.yticks(np.arange(len(top_feats)), top_feats)
    # xt = np.linspace(0, sub.shape[0]-1, min(10, sub.shape[0]), dtype=int)
    # plt.xticks(xt)
    # plt.xlabel("Time step within window")
    # plt.title("Per-timestep contributions for top features")
    # plt.tight_layout()
    # plt.show()

idx_slider   = widgets.IntSlider(min=0, max=len(X_test)-1, step=1, value=0, description="Test idx")
topk_slider  = widgets.IntSlider(min=3, max=min(20, F), step=1, value=8, description="Top-K")
agg_dropdown = widgets.Dropdown(
    options=[("Signed (↑/↓ risk)", "signed"), ("Magnitude (importance)", "magnitude")],
    value="signed", description="Aggregate"
)

ui  = widgets.VBox([idx_slider, topk_slider, agg_dropdown])
out = widgets.interactive_output(
    predict_and_explain, {"i": idx_slider, "top_k": topk_slider, "aggregation": agg_dropdown}
)
display(ui, out)


In [ ]:
# ----- RUN THIS IN YOUR NOTEBOOK/ENV WHERE model/X_train/X_test/feature_cols EXIST -----
import numpy as np, json, textwrap
from pathlib import Path

# --- Helpers copied from your widget code ---
def _to_np(x):
    import tensorflow as tf
    if tf.is_tensor(x):
        try:
            return x.numpy()
        except Exception:
            return tf.convert_to_tensor(x).numpy()
    return np.asarray(x)

X_train = _to_np(X_train).astype(np.float32)
X_test  = _to_np(X_test).astype(np.float32)
assert X_train.ndim == 3 and X_test.ndim == 3, "Expected (N, T, F) arrays."
T, F = X_test.shape[1], X_test.shape[2]
feat_names = list(feature_cols)
assert len(feat_names) == F, f"feature_cols length {len(feature_cols)} != F={F}"

# --- Predict probabilities for all test samples ---
probs = model.predict(X_test, verbose=0).reshape(-1).clip(0, 1)  # (N,)

# --- Compute SHAP (KernelExplainer) once, then explain all samples ---
#     We aggregate over time to get feature-level contributions.
import shap
from functools import lru_cache

def _flatten(x):
    xx = np.asarray(x, dtype=np.float32)
    return xx.reshape(xx.shape[0], -1)

def _predict_from_flat(X_flat):
    X_flat = np.asarray(X_flat, dtype=np.float32)
    if X_flat.ndim == 1:
        X_flat = X_flat[None, :]
    B = X_flat.shape[0]
    x = X_flat.reshape(B, T, F)
    return np.asarray(model.predict(x, verbose=0)).reshape(-1)

@lru_cache(maxsize=1)
def _get_kernel_explainer(bg_size=24, seed=42):
    rng = np.random.default_rng(seed)
    n = min(bg_size, len(X_train))
    bg_idx = rng.choice(len(X_train), size=n, replace=False)
    bg_flat = _flatten(X_train[bg_idx])  # (n, T*F)
    return shap.KernelExplainer(_predict_from_flat, bg_flat)

# You can cap N_MAX if runtime/payload is a concern (set to None for all)
N = len(X_test)
N_MAX = 120  # e.g., 1000
use_N = min(N, N_MAX or N)

expl = _get_kernel_explainer()

shap_feat_signed = np.zeros((use_N, F), dtype=np.float32)
shap_feat_mag    = np.zeros((use_N, F), dtype=np.float32)

# nsamples is the main speed/quality lever (100–300 typical for demos)
nsamples = 96

for i in range(use_N):
    x = X_test[i:i+1]                # (1, T, F)
    sv = expl.shap_values(_flatten(x), nsamples=nsamples)
    if isinstance(sv, list):
        sv = sv[0]
    sv = np.asarray(sv, dtype=np.float32).reshape(T, F)  # (T, F)

    # Aggregate over time to feature-level contributions
    signed = sv.sum(axis=0)          # shape (F,)
    mag    = np.abs(sv).sum(axis=0)  # shape (F,)

    shap_feat_signed[i] = signed
    shap_feat_mag[i]    = mag

# --- Build a single self-contained HTML page (no backend) ---
# NOTE: We embed arrays directly. For very large N, consider writing out JSON files and fetching via <script> fetch().
html = f"""<!doctype html>
<html lang="en">
<head>
  <meta charset="utf-8" />
  <meta name="viewport" content="width=device-width,initial-scale=1" />
  <title>Aggression Risk + Explanations (Static)</title>
  <link rel="preconnect" href="https://cdn.jsdelivr.net" />
  <script src="https://cdn.jsdelivr.net/npm/chart.js"></script>
  <style>
    :root {{
      --bg:#fafafa; --card:#fff; --fg:#111; --muted:#666; --accent:#2dd4bf; --accent2:#6aa6ff;
    }}
    body {{
      font-family: system-ui, -apple-system, Segoe UI, Roboto, Helvetica, Arial, sans-serif;
      margin: 2rem; background: var(--bg); color: var(--fg);
    }}
    .card {{
      max-width: 980px; margin: 0 auto; background: var(--card); border-radius: 16px;
      box-shadow: 0 6px 20px rgba(0,0,0,0.08); padding: 24px 22px;
    }}
    h1 {{ font-size: 1.25rem; margin: 0 0 .25rem; }}
    .sub {{ color: var(--muted); margin-bottom: 1rem; }}
    .row {{ display:flex; gap:16px; align-items:center; flex-wrap:wrap; }}
    .col {{ flex:1 1 340px; min-width: 280px; }}
    .pill {{ display:inline-block; padding: 6px 10px; border-radius:999px; background:#f1f3f5; font-weight:600; }}
    input[type="range"] {{ width: 100%; }}
    select, input[type="number"] {{ padding:6px 8px; border-radius:8px; border:1px solid #ddd; }}
    .meter-wrap {{ position:relative; height:14px; background:#eee; border-radius:999px; overflow:hidden; }}
    .meter-bar {{ position:absolute; left:0; top:0; bottom:0; width:0%; background:linear-gradient(90deg, var(--accent2), var(--accent)); }}
    .muted {{ color:var(--muted); font-size:.95rem; }}
    .kpi {{ font-size:2.25rem; font-weight:800; }}
    .footer {{ margin-top: 1rem; color:#777; font-size:.85rem; }}
  </style>
</head>
<body>
  <div class="card">
    <h1>Prototype Early-Warning: 24h Aggression Risk (Static)</h1>
    <div class="sub">Precomputed probabilities and SHAP feature contributions (aggregated over time). No backend required.</div>

    <div class="row">
      <div class="col">
        <label for="idx"><strong>Test index</strong></label>
        <input id="idx" type="range" min="0" max="{use_N-1}" value="0" step="1" />
        <div class="pill" id="idxVal">0</div>
      </div>
      <div class="col">
        <label for="agg"><strong>Aggregation</strong></label><br/>
        <select id="agg">
          <option value="signed" selected>Signed (↑/↓ risk)</option>
          <option value="magnitude">Magnitude (importance)</option>
        </select>
      </div>
      <div class="col">
        <label for="topk"><strong>Top-K features</strong></label><br/>
        <input id="topk" type="number" min="3" max="{F}" value="{min(8, F)}" />
      </div>
    </div>

    <div class="row" style="margin-top:18px;">
      <div class="col">
        <div class="muted">Predicted probability of a violent incident in next 24 hours</div>
        <div class="kpi" id="probText">0.00%</div>
        <div class="meter-wrap" style="margin-top:8px;"><div class="meter-bar" id="meter"></div></div>
      </div>
      <div class="col">
        <div class="muted">Ranked top features</div>
        <ul id="featList" style="margin:6px 0 0 18px;"></ul>
      </div>
    </div>

    <div style="margin-top: 16px;">
      <canvas id="barChart" height="240"></canvas>
    </div>

    <div class="footer">
      Notes: SHAP values were computed offline using a KernelExplainer with a small background set, then aggregated over the time axis to per-feature contributions. This static page embeds arrays as JSON.
    </div>
  </div>

  <script>
    // --- Embedded data (JSON) ---
    const PROBS = {probs[:use_N].round(6).tolist()};
    const FEAT_NAMES = {json.dumps(list(map(str, feat_names)))};
    const SHAP_SIGNED = {np.round(shap_feat_signed[:use_N], 6).tolist()};
    const SHAP_MAG    = {np.round(shap_feat_mag[:use_N], 6).tolist()};

    // --- DOM elements ---
    const idx   = document.getElementById('idx');
    const idxVal= document.getElementById('idxVal');
    const agg   = document.getElementById('agg');
    const topk  = document.getElementById('topk');
    const probText = document.getElementById('probText');
    const meter = document.getElementById('meter');
    const featList = document.getElementById('featList');

    // --- Chart.js setup ---
    const ctx = document.getElementById('barChart').getContext('2d');
    let barChart = new Chart(ctx, {{
      type: 'bar',
      data: {{
        labels: [],
        datasets: [{{
          label: 'Contribution',
          data: [],
        }}]
      }},
      options: {{
        responsive: true,
        maintainAspectRatio: false,
        indexAxis: 'y',
        scales: {{
          x: {{
            grid: {{ display: true }},
            ticks: {{ callback: (v) => v }}
          }},
          y: {{
            grid: {{ display: false }}
          }}
        }},
        plugins: {{
          legend: {{ display: false }},
          tooltip: {{ enabled: true }}
        }}
      }}
    }});

    function render(i) {{
      i = Math.max(0, Math.min(PROBS.length-1, i));
      idxVal.textContent = i;
      const p = Math.max(0, Math.min(1, PROBS[i] ?? 0));
      probText.textContent = (p * 100).toFixed(2) + '%';
      meter.style.width = (p * 100).toFixed(2) + '%';

      const mode = agg.value;
      const vec = (mode === 'signed' ? SHAP_SIGNED[i] : SHAP_MAG[i]).slice(); // copy
      // rank by absolute value (so large negatives also rank high); keep sign for display
      const order = vec.map((v, j) => [j, v, Math.abs(v)]).sort((a,b) => b[2]-a[2]);
      const k = Math.max(1, Math.min(parseInt(topk.value || 8), FEAT_NAMES.length));
      const sel = order.slice(0, k);

      const labels = sel.map(x => FEAT_NAMES[x[0]]);
      const values = sel.map(x => x[1]);

      // Update text list
      featList.innerHTML = '';
      sel.forEach(([j, v]) => {{
        const li = document.createElement('li');
        const sign = v >= 0 ? '↑' : '↓';
        li.textContent = `${{FEAT_NAMES[j]}}: ${{v.toFixed(4)}} ${{sign}}`;
        featList.appendChild(li);
      }});

      // Update chart
      barChart.data.labels = labels;
      barChart.data.datasets[0].data = values;
      barChart.update();
    }}

    idx.addEventListener('input', () => render(Number(idx.value)));
    agg.addEventListener('change', () => render(Number(idx.value)));
    topk.addEventListener('change', () => render(Number(idx.value)));

    // initial render
    render(0);
  </script>
</body>
</html>
"""

Path("risk_explain_widget.html").write_text(html, encoding="utf-8")
print("Wrote risk_explain_widget.html")


In [ ]:
import os; os.environ["TQDM_DISABLE"] = "1"
from functools import lru_cache
from IPython.display import clear_output

@lru_cache(maxsize=512)
def _explain_cached(i, aggregation, nsamples, bg_size_key=32):
    """
    Cached wrapper so moving the slider back-and-forth is instant.
    bg_size_key is just to invalidate cache if you change background size.
    """
    x = X_test[i:i+1]
    prob = float(model.predict(x, verbose=0)[0][0])

    expl = _get_kernel_explainer()  # uses cached explainer
    sv = expl.shap_values(_flatten(x), nsamples=nsamples)
    if isinstance(sv, list):
        sv = sv[0]
    shap_2d = np.asarray(sv, dtype=np.float32).reshape(T, F)

    if aggregation == "magnitude":
        feat_contrib = np.abs(shap_2d).sum(axis=0)
    else:
        feat_contrib = shap_2d.sum(axis=0)

    return prob, feat_contrib, shap_2d

_RUNNING = False

def predict_and_explain(i, top_k=8, aggregation="signed", nsamples=200):
    global _RUNNING
    if _RUNNING:
        return
    _RUNNING = True
    try:
        clear_output(wait=True)

        prob, feat_contrib, shap_2d = _explain_cached(i, aggregation, nsamples)

        order = np.argsort(np.abs(feat_contrib))[::-1][:top_k]
        top_feats = [feat_names[j] for j in order]
        top_vals  = feat_contrib[order]

        plt.figure(figsize=(6, 4))
        plt.title(f"Predicted risk (next 24h): {prob:.2%}")
        plt.axis("off")
        plt.show()

        plt.figure(figsize=(7, max(3, 0.35*top_k)))
        y = np.arange(len(top_feats))
        plt.barh(y, top_vals, align="center")
        plt.yticks(y, top_feats)
        plt.xlabel("Contribution" if aggregation=="signed" else "Importance (|SHAP|)")
        plt.title(f"Top {top_k} features ({aggregation})")
        plt.gca().invert_yaxis()
        plt.grid(True, axis="x", linestyle="--", alpha=0.35)
        plt.tight_layout()
        plt.show()

        sub = shap_2d[:, order]
        plt.figure(figsize=(8, max(3, 0.35*top_k)))
        im = plt.imshow(sub.T, aspect="auto", interpolation="nearest")
        plt.colorbar(im, label="SHAP value (signed)")
        plt.yticks(np.arange(len(top_feats)), top_feats)
        xt = np.linspace(0, sub.shape[0]-1, min(10, sub.shape[0]), dtype=int)
        plt.xticks(xt)
        plt.xlabel("Time step within window")
        plt.title("Per-timestep contributions for top features")
        plt.tight_layout()
        plt.show()

    finally:
        _RUNNING = False
